# Mining - Train Equipment Failure Model

Trains a GradientBoostingClassifier on the equipment failure Delta table, evaluates performance, and logs everything to MLflow.

In [ ]:
%run ../_setup

In [ ]:
import tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split

from shared.config import RANDOM_SEED, TEST_SIZE
from shared.pipelines import build_classifier_pipeline
from shared.evaluation import evaluate_classifier, print_evaluation_summary, get_feature_importance
from shared.plotting import plot_confusion_matrix, plot_roc_curve, plot_feature_importance

from mining.src.preprocess import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COLUMN,
    TARGET_NAMES,
)

In [ ]:
INDUSTRY = "mining"
MODEL_DISPLAY_NAME = "Mining Equipment Failure"
CLASSIFIER_KWARGS = {"random_state": RANDOM_SEED, "n_estimators": 250, "max_depth": 5}

In [ ]:
df = read_from_delta(INDUSTRY)
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]
print(f"Features: {X.shape[1]}, Samples: {len(X):,}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
print(f"Train: {len(X_train):,}, Test: {len(X_test):,}")

In [ ]:
cfg = INDUSTRY_CONFIG[INDUSTRY]
mlflow.set_experiment(cfg["experiment_path"])
print(f"MLflow experiment: {cfg['experiment_path']}")

In [ ]:
with mlflow.start_run(run_name=MODEL_DISPLAY_NAME) as run:

    # -- Build and fit --------------------------------------------------------
    pipeline = build_classifier_pipeline(
        numeric_features=NUMERIC_FEATURES,
        categorical_features=CATEGORICAL_FEATURES,
        **CLASSIFIER_KWARGS,
    )
    pipeline.fit(X_train, y_train)

    # -- Evaluate -------------------------------------------------------------
    metrics = evaluate_classifier(pipeline, X_test, y_test, target_names=TARGET_NAMES)
    importance = get_feature_importance(pipeline)
    print_evaluation_summary(metrics, model_name=MODEL_DISPLAY_NAME)

    # -- Log params -----------------------------------------------------------
    mlflow.log_param("industry", INDUSTRY)
    mlflow.log_param("n_samples", len(df))
    mlflow.log_param("n_train", len(X_train))
    mlflow.log_param("n_test", len(X_test))
    mlflow.log_param("test_size", TEST_SIZE)
    mlflow.log_param("random_seed", RANDOM_SEED)
    mlflow.log_param("numeric_features", ", ".join(NUMERIC_FEATURES))
    mlflow.log_param("categorical_features", ", ".join(CATEGORICAL_FEATURES))

    clf_params = pipeline.named_steps["classifier"].get_params()
    for key, value in clf_params.items():
        mlflow.log_param(f"clf_{key}", value)

    # -- Log metrics ----------------------------------------------------------
    mlflow.log_metric("accuracy", metrics["accuracy"])
    mlflow.log_metric("precision", metrics["precision"])
    mlflow.log_metric("recall", metrics["recall"])
    mlflow.log_metric("f1", metrics["f1"])
    mlflow.log_metric("roc_auc", metrics["roc_auc"])

    report_dict = metrics["classification_report_dict"]
    for class_name in TARGET_NAMES:
        class_metrics = report_dict[class_name]
        safe_name = class_name.lower().replace(" ", "_")
        mlflow.log_metric(f"{safe_name}_precision", class_metrics["precision"])
        mlflow.log_metric(f"{safe_name}_recall", class_metrics["recall"])
        mlflow.log_metric(f"{safe_name}_f1", class_metrics["f1-score"])

    # -- Log artefacts (plots) ------------------------------------------------
    with tempfile.TemporaryDirectory() as tmp_dir:
        plots_dir = Path(tmp_dir) / "plots"
        plots_dir.mkdir()

        plot_confusion_matrix(
            metrics["confusion_matrix"],
            target_names=TARGET_NAMES,
            output_path=plots_dir / "confusion_matrix.png",
            title=f"{MODEL_DISPLAY_NAME} - Confusion Matrix",
        )
        plot_roc_curve(
            y_test,
            metrics["y_proba"],
            output_path=plots_dir / "roc_curve.png",
            title=f"{MODEL_DISPLAY_NAME} - ROC Curve",
        )
        plot_feature_importance(
            importance,
            output_path=plots_dir / "feature_importance.png",
            title=f"{MODEL_DISPLAY_NAME} - Feature Importance",
        )

        mlflow.log_artifacts(str(plots_dir), artifact_path="plots")

    # -- Log model ------------------------------------------------------------
    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="model",
        registered_model_name=cfg["registered_model_name"],
    )

    # -- Tags -----------------------------------------------------------------
    mlflow.set_tag("industry", INDUSTRY)
    mlflow.set_tag("model_type", "GradientBoostingClassifier")
    mlflow.set_tag("delta_table", get_delta_table_name(INDUSTRY))

    print(f"\nMLflow run ID: {run.info.run_id}")

In [ ]:
print(f"\n{'='*60}")
print(f"  {MODEL_DISPLAY_NAME} - Results Summary")
print(f"{'='*60}")
print(f"  Industry         : {INDUSTRY}")
print(f"  Samples          : {len(df):,}")
print(f"  Train / Test     : {len(X_train):,} / {len(X_test):,}")
print(f"  Accuracy         : {metrics['accuracy']:.4f}")
print(f"  Precision        : {metrics['precision']:.4f}")
print(f"  Recall           : {metrics['recall']:.4f}")
print(f"  F1               : {metrics['f1']:.4f}")
print(f"  ROC AUC          : {metrics['roc_auc']:.4f}")
print(f"  MLflow Experiment: {cfg['experiment_path']}")
print(f"  Registered Model : {cfg['registered_model_name']}")
print(f"{'='*60}")